# Märklin DC Motor PWM & BEMF Simulation Engine

This notebook simulates the electromechanical dynamic behavior of DC motors (e.g., Märklin model train motors) driven by PWM, incorporating Back-EMF (bEMF) sensing gaps and commutator ripple detection.

- **Closed-Loop PID Control**: Option to enable closed-loop motor speed regulation where feedback is calculated from the mean of the last 25% (last quarter) of bEMF ADC samples collected inside each measurement gap.

In [1]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 1. Core Physics Simulation Model

The function  models the coupled electrical and mechanical differential equations for a DC motor under PWM excitation with high-impedance bEMF measurement gaps.

In [2]:
import numpy as np

def simulate_dc_motor(
    n_coils=3,
    v_in=15.0,
    pwm_freq=1000.0,
    pwm_duty=50.0,
    pwm_duration_ms=200.0,
    bemf_window_ms=2.0,
    bemf_sample_rate_hz=50000.0,
    gap_interval_ms=50.0,
    R=8.0,
    L=0.005,
    Ke=0.015,
    Kt=0.015,
    J=1e-5,
    B=1e-5,
    T_coulomb=0.001,
    ripple_amplitude=0.08,
    comm_spike_amplitude=0.20,
    comm_spike_width=0.10,
    brush_noise_amplitude=0.02,
    dt=2e-6,
    enable_pid=False,
    target_bemf=5.0,
    kp=10.0,
    ki=50.0,
    kd=0.1
):
    """
    Simulates DC motor electromechanical dynamics under PWM drive with periodic bEMF gaps,
    realistic commutator current spikes, brush arcing noise, and optional closed-loop PID
    speed control calculated from the last quarter (last 25%) of bEMF readout samples inside each gap.
    """
    total_time = pwm_duration_ms / 1000.0
    num_steps = int(total_time / dt)
    
    t_arr = np.linspace(0, total_time, num_steps)
    v_m_arr = np.zeros(num_steps)
    i_arr = np.zeros(num_steps)
    w_arr = np.zeros(num_steps)
    theta_arr = np.zeros(num_steps)
    duty_arr = np.zeros(num_steps)
    is_gap_arr = np.zeros(num_steps, dtype=bool)
    comm_spike_arr = np.zeros(num_steps)
    
    i = 0.0
    w = 0.0
    theta = 0.0
    
    pwm_period = 1.0 / pwm_freq
    gap_interval = gap_interval_ms / 1000.0
    bemf_window = bemf_window_ms / 1000.0
    comm_period = np.pi / n_coils
    bemf_adc_dt = 1.0 / bemf_sample_rate_hz
    
    active_duty = pwm_duty
    integral_error = 0.0
    prev_error = 0.0
    
    adc_timestamps = []
    adc_bemf_readings = []
    gap_metrics = []
    
    in_gap = False
    gap_start_idx = 0
    gap_adc_times = []
    gap_adc_readings = []
    next_adc_time = 0.0
    gap_counter = 0
    
    for idx, t in enumerate(t_arr):
        time_in_gap_cycle = t % gap_interval
        is_gap = (time_in_gap_cycle < bemf_window) and (t > 0.005) # start gaps after initial startup
        is_gap_arr[idx] = is_gap
        
        # Gap entry detection
        if is_gap and not in_gap:
            in_gap = True
            gap_start_idx = idx
            gap_adc_times = []
            gap_adc_readings = []
            next_adc_time = t
            gap_counter += 1
            
        # Gap exit detection: process samples & perform PID update
        if in_gap and not is_gap:
            in_gap = False
            t_gap_start = t_arr[gap_start_idx]
            t_gap_end = t
            q4_start_t = t_gap_start + 0.75 * bemf_window
            
            gap_adc_t_arr = np.array(gap_adc_times)
            gap_adc_b_arr = np.array(gap_adc_readings)
            
            q4_mask = gap_adc_t_arr >= q4_start_t
            if np.any(q4_mask):
                q4_adc_t = gap_adc_t_arr[q4_mask]
                q4_adc_b = gap_adc_b_arr[q4_mask]
                measured_bemf = float(np.mean(q4_adc_b))
            elif len(gap_adc_b_arr) > 0:
                q4_adc_t = gap_adc_t_arr[-1:]
                q4_adc_b = gap_adc_b_arr[-1:]
                measured_bemf = float(gap_adc_b_arr[-1])
            else:
                q4_adc_t = np.array([])
                q4_adc_b = np.array([])
                measured_bemf = 0.0
                
            if enable_pid:
                dt_pid = gap_interval
                error = target_bemf - measured_bemf
                integral_error += error * dt_pid
                if ki > 0:
                    max_i = 100.0 / ki
                    integral_error = float(np.clip(integral_error, -max_i, max_i))
                deriv_error = (error - prev_error) / dt_pid
                prev_error = error
                
                unclamped = kp * error + ki * integral_error + kd * deriv_error
                active_duty = float(np.clip(unclamped, 0.0, 100.0))
                
            gap_metrics.append({
                'gap_index': gap_counter,
                't_start': t_gap_start,
                't_end': t_gap_end,
                'q4_start_t': q4_start_t,
                'adc_t': gap_adc_t_arr,
                'adc_bemf': gap_adc_b_arr,
                'q4_adc_t': q4_adc_t,
                'q4_adc_bemf': q4_adc_b,
                'measured_bemf': measured_bemf,
                'duty_applied': active_duty
            })

        duty_arr[idx] = active_duty
        
        if is_gap:
            v_applied = 0.0
            hbridge_state = 'FLOAT'
        else:
            time_in_pwm_cycle = t % pwm_period
            pwm_on_time = (active_duty / 100.0) * pwm_period
            if time_in_pwm_cycle < pwm_on_time:
                v_applied = v_in
                hbridge_state = 'DRIVE'
            else:
                v_applied = 0.0
                hbridge_state = 'OFF'
        
        # Commutator angular position calculation
        phi = (theta % comm_period) / comm_period
        dist_to_trans = np.minimum(phi, 1.0 - phi)
        comm_spike = comm_spike_amplitude * np.exp(- (dist_to_trans / (comm_spike_width / 2.0))**2) if comm_spike_width > 0 else 0.0
        comm_spike_arr[idx] = comm_spike
        brush_noise = brush_noise_amplitude * (0.6 * np.sin(127.0 * theta) + 0.4 * np.cos(311.0 * t))
        
        ripple_freq_factor = 2.0 * n_coils
        ripple_mod = 1.0 + ripple_amplitude * np.cos(ripple_freq_factor * theta)
        
        # Commutation bridging reduces effective resistance & bEMF temporarily during coil shorting
        spike_factor = (comm_spike / comm_spike_amplitude) if comm_spike_amplitude > 0 else 0.0
        e_back = Ke * w * (ripple_mod - 0.25 * spike_factor)
        R_eff = R * (1.0 + 0.05 * np.sin(ripple_freq_factor * theta) - 0.15 * spike_factor)
        
        if hbridge_state == 'DRIVE':
            v_motor = v_applied
            di_dt = (v_motor - e_back - i * R_eff) / L
        elif hbridge_state in ('OFF', 'FLOAT'):
            if i > 0.0005:
                v_motor = -0.7
                di_dt = (v_motor - e_back - i * R_eff) / L
            else:
                i = 0.0
                v_motor = e_back
                di_dt = 0.0
                
        i += di_dt * dt
        if i < 0:
            i = 0.0
            
        T_elec = Kt * i * ripple_mod
        T_fric = B * w + (T_coulomb if w > 0.001 else 0.0)
        dw_dt = (T_elec - T_fric) / J
        
        w += dw_dt * dt
        if w < 0:
            w = 0.0
            
        theta += w * dt
        
        # Terminal current measured at shunt includes brush commutation surge & contact noise
        i_measured = i * (1.0 + comm_spike + brush_noise) if i > 0.0001 else 0.0
        
        v_m_arr[idx] = v_motor
        i_arr[idx] = i_measured
        w_arr[idx] = w
        theta_arr[idx] = theta
        
        # Sample ADC readings during gaps
        if in_gap and is_gap:
            if t >= next_adc_time:
                gap_adc_times.append(t)
                gap_adc_readings.append(v_motor)
                adc_timestamps.append(t)
                adc_bemf_readings.append(v_motor)
                next_adc_time += bemf_adc_dt

    return {
        't': t_arr,
        'v_m': v_m_arr,
        'i': i_arr,
        'w': w_arr,
        'theta': theta_arr,
        'duty': duty_arr,
        'is_gap': is_gap_arr,
        'comm_spike': comm_spike_arr,
        'adc_t': np.array(adc_timestamps),
        'adc_bemf': np.array(adc_bemf_readings),
        'gap_metrics': gap_metrics
    }


## 2. Motor Characteristics Presets

Pre-configured parameter sets tailored for various locomotive motor types (Alte Märklinmotoren, Glockenankermotor, Moderne Motoren, and Grosse Gartenbahnmotoren).

In [3]:
MOTOR_PRESETS = {
    'Alte Märklinmotoren (3-Pol LFCM/SFCM/DCM)': {
        'n_coils': 3,
        'v_in': 15.0,
        'pwm_freq': 1000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 7.5,
        'L': 0.012,
        'Ke': 0.018,
        'Kt': 0.018,
        'J': 3e-5,
        'B': 2e-5,
        'T_coulomb': 0.002,
        'ripple_amplitude': 0.12,
        'comm_spike_amplitude': 0.35,
        'comm_spike_width': 0.12,
        'brush_noise_amplitude': 0.04,
        'description': 'Classic heavy iron armature Märklin motor with high inductance, pronounced commutator ripple, and high brush commutation spikes.'
    },
    'Glockenankermotor (Faulhaber / Maxon)': {
        'n_coils': 5,
        'v_in': 12.0,
        'pwm_freq': 16000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 1.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 15.0,
        'L': 0.0003,
        'Ke': 0.012,
        'Kt': 0.012,
        'J': 2e-7,
        'B': 3e-6,
        'T_coulomb': 0.0002,
        'ripple_amplitude': 0.03,
        'comm_spike_amplitude': 0.08,
        'comm_spike_width': 0.05,
        'brush_noise_amplitude': 0.005,
        'description': 'Coreless motor with ultra-low inductance, smooth precious metal commutation, and minimal arcing.'
    },
    'Moderne Motoren (5-Pol HO DC Can)': {
        'n_coils': 5,
        'v_in': 15.0,
        'pwm_freq': 4000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 12.0,
        'L': 0.003,
        'Ke': 0.015,
        'Kt': 0.015,
        'J': 3e-6,
        'B': 1e-5,
        'T_coulomb': 0.0008,
        'ripple_amplitude': 0.06,
        'comm_spike_amplitude': 0.20,
        'comm_spike_width': 0.08,
        'brush_noise_amplitude': 0.02,
        'description': 'Standard modern 5-pole enclosed DC can motor used in current locomotives.'
    },
    'Grosse Gartenbahnmotoren (LGB / Spur I)': {
        'n_coils': 5,
        'v_in': 18.0,
        'pwm_freq': 1000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.5,
        'bemf_sample_rate_hz': 50000.0,
        'R': 4.0,
        'L': 0.020,
        'Ke': 0.035,
        'Kt': 0.035,
        'J': 1.5e-4,
        'B': 5e-5,
        'T_coulomb': 0.005,
        'ripple_amplitude': 0.08,
        'comm_spike_amplitude': 0.25,
        'comm_spike_width': 0.10,
        'brush_noise_amplitude': 0.03,
        'description': 'Large, heavy motor for garden railways (G scale / Gauge 1) with high torque, heavy current surges, and strong commutation spikes.'
    }
}

## 3. Interactive UI & Simulation Plotting Dashboard

Use the controls below to select motor profiles, adjust operating and physical parameters, and **jump between individual bEMF readout gaps** to analyze edge cases like inductive freewheeling tail saturation and commutation spike overlaps. You can also enable closed-loop PID control based on the last quarter of bEMF readouts during each gap window to observe dynamic speed stabilization and duty cycle response.

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# Default controls initialization
preset_dropdown = widgets.Dropdown(
    options=list(MOTOR_PRESETS.keys()),
    value='Alte Märklinmotoren (3-Pol LFCM/SFCM/DCM)',
    description='Preset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

w_n_coils = widgets.IntSlider(value=3, min=3, max=12, step=1, description='Coils (n):', style={'description_width': 'initial'})
w_v_in = widgets.FloatSlider(value=15.0, min=3.0, max=24.0, step=0.5, description='V_in (V):', style={'description_width': 'initial'})
w_pwm_freq = widgets.FloatLogSlider(value=1000.0, base=10, min=2, max=4.5, step=0.05, description='PWM Freq (Hz):', style={'description_width': 'initial'})
w_pwm_duty = widgets.FloatSlider(value=50.0, min=0.0, max=100.0, step=1.0, description='Base Duty (%):', style={'description_width': 'initial'})
w_pwm_duration = widgets.FloatSlider(value=200.0, min=20.0, max=20000.0, step=10.0, description='Duration (ms):', style={'description_width': 'initial'})
w_bemf_window = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.1, description='bEMF Window (ms):', style={'description_width': 'initial'})
w_bemf_rate = widgets.FloatLogSlider(value=50000.0, base=10, min=3, max=5.3, step=0.1, description='ADC Rate (Hz):', style={'description_width': 'initial'})

w_R = widgets.FloatSlider(value=8.0, min=1.0, max=30.0, step=0.5, description='R (Ω):', style={'description_width': 'initial'})
w_L = widgets.FloatSlider(value=5.0, min=0.1, max=20.0, step=0.1, description='L (mH):', style={'description_width': 'initial'})
w_Ke = widgets.FloatSlider(value=0.015, min=0.001, max=0.050, step=0.001, readout_format='.3f', description='Ke (V/(rad/s)):', style={'description_width': 'initial'})
w_J = widgets.FloatSlider(value=10.0, min=1.0, max=100.0, step=1.0, description='J (g·cm²):', style={'description_width': 'initial'})
w_ripple = widgets.FloatSlider(value=0.08, min=0.0, max=0.30, step=0.01, description='Ripple Amp:', style={'description_width': 'initial'})
w_comm_spike_amp = widgets.FloatSlider(value=0.20, min=0.0, max=0.80, step=0.02, description='Comm Spike Amp:', style={'description_width': 'initial'})
w_comm_spike_width = widgets.FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description='Comm Spike Width:', style={'description_width': 'initial'})
w_brush_noise = widgets.FloatSlider(value=0.02, min=0.0, max=0.10, step=0.005, readout_format='.3f', description='Brush Noise Amp:', style={'description_width': 'initial'})

# PID Controls
w_enable_pid = widgets.Checkbox(value=False, description='Enable PID Control', style={'description_width': 'initial'})
w_target_bemf = widgets.FloatSlider(value=2.5, min=0.1, max=15.0, step=0.1, description='Target bEMF (V):', style={'description_width': 'initial'})
w_kp = widgets.FloatText(value=15.0, description='Kp:', layout=widgets.Layout(width='160px'), style={'description_width': 'initial'})
w_ki = widgets.FloatText(value=80.0, description='Ki:', layout=widgets.Layout(width='160px'), style={'description_width': 'initial'})
w_kd = widgets.FloatText(value=0.1, description='Kd:', layout=widgets.Layout(width='160px'), style={'description_width': 'initial'})

w_gap_index = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description='Readout Gap #:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

description_out = widgets.HTML(value=f"<b>Description:</b> {MOTOR_PRESETS[preset_dropdown.value]['description']}")
gap_info_out = widgets.HTML(value="<b>Gap Status:</b> Run simulation to inspect bEMF readout gaps.")

def on_preset_change(change):
    p = MOTOR_PRESETS[change['new']]
    w_n_coils.value = p['n_coils']
    w_v_in.value = p['v_in']
    w_pwm_freq.value = p['pwm_freq']
    w_pwm_duty.value = p['pwm_duty']
    w_pwm_duration.value = p['pwm_duration_ms']
    w_bemf_window.value = p['bemf_window_ms']
    w_bemf_rate.value = p['bemf_sample_rate_hz']
    w_R.value = p['R']
    w_L.value = p['L'] * 1000.0
    w_Ke.value = p['Ke']
    w_J.value = p['J'] * 1e6
    w_ripple.value = p['ripple_amplitude']
    w_comm_spike_amp.value = p.get('comm_spike_amplitude', 0.20)
    w_comm_spike_width.value = p.get('comm_spike_width', 0.10)
    w_brush_noise.value = p.get('brush_noise_amplitude', 0.02)
    description_out.value = f"<b>Description:</b> {p['description']}"

preset_dropdown.observe(on_preset_change, names='value')

out = widgets.Output()

cached_sim = {'params': None, 'res': None}

def run_and_plot_simulation(b=None):
    with out:
        clear_output(wait=True)
        
        current_params = (
            w_n_coils.value, w_v_in.value, w_pwm_freq.value, w_pwm_duty.value,
            w_pwm_duration.value, w_bemf_window.value, w_bemf_rate.value,
            w_R.value, w_L.value, w_Ke.value, w_J.value, w_ripple.value,
            w_comm_spike_amp.value, w_comm_spike_width.value, w_brush_noise.value,
            w_enable_pid.value, w_target_bemf.value, w_kp.value, w_ki.value, w_kd.value
        )
        
        if cached_sim['params'] == current_params and cached_sim['res'] is not None:
            res = cached_sim['res']
        else:
            res = simulate_dc_motor(
                n_coils=w_n_coils.value,
                v_in=w_v_in.value,
                pwm_freq=w_pwm_freq.value,
                pwm_duty=w_pwm_duty.value,
                pwm_duration_ms=w_pwm_duration.value,
                bemf_window_ms=w_bemf_window.value,
                bemf_sample_rate_hz=w_bemf_rate.value,
                R=w_R.value,
                L=w_L.value / 1000.0,
                Ke=w_Ke.value,
                Kt=w_Ke.value,
                J=w_J.value / 1e6,
                ripple_amplitude=w_ripple.value,
                comm_spike_amplitude=w_comm_spike_amp.value,
                comm_spike_width=w_comm_spike_width.value,
                brush_noise_amplitude=w_brush_noise.value,
                enable_pid=w_enable_pid.value,
                target_bemf=w_target_bemf.value,
                kp=w_kp.value,
                ki=w_ki.value,
                kd=w_kd.value
            )
            cached_sim['params'] = current_params
            cached_sim['res'] = res
        
        t_ms = res['t'] * 1000.0
        v_m = res['v_m']
        i_m = res['i']
        rpm = res['w'] * 60.0 / (2.0 * np.pi)
        duty = res['duty']
        
        # Identify discrete bEMF gap windows
        is_gap = res['is_gap']
        diff = np.diff(is_gap.astype(int))
        gap_starts = np.where(diff == 1)[0] + 1
        if is_gap[0]:
            gap_starts = np.insert(gap_starts, 0, 0)
        gap_ends = np.where(diff == -1)[0]
        if is_gap[-1]:
            gap_ends = np.append(gap_ends, len(is_gap) - 1)
            
        num_gaps = len(gap_starts)
        
        if num_gaps > 0:
            w_gap_index.disabled = False
            w_gap_index.max = num_gaps
            selected_gap = min(max(1, w_gap_index.value), num_gaps)
            if w_gap_index.value != selected_gap:
                w_gap_index.value = selected_gap
            g_idx = selected_gap - 1
        else:
            w_gap_index.disabled = True
            selected_gap = 1
            g_idx = 0
            
        fig, axs = plt.subplots(3, 1, figsize=(11, 10), sharex=False)
        fig.suptitle('Märklin DC Motor PWM Drive & bEMF Readout Simulation', fontsize=14, fontweight='bold')
        
        # Plot 1: Overall Overview (Voltage & Current & Readout Gap Locations)
        ax1 = axs[0]
        ax1.plot(t_ms, v_m, label='Motor Terminal Voltage (V)', color='tab:blue', alpha=0.8)
        ax1.set_ylabel('Voltage (V)', color='tab:blue')
        title_mode = "Closed-Loop PID Control" if w_enable_pid.value else "Open-Loop PWM Drive"
        ax1.set_title(f'Global Overview: Voltage, Current, and Readout Gap Locations ({title_mode})')
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        ax1_i = ax1.twinx()
        ax1_i.plot(t_ms, i_m, label='Current (A)', color='tab:orange', alpha=0.7)
        ax1_i.set_ylabel('Current (A)', color='tab:orange')
        
        # Shade all gap intervals & highlight selected gap
        if num_gaps > 0:
            for k in range(num_gaps):
                t_g_start = res['t'][gap_starts[k]] * 1000.0
                t_g_end = res['t'][gap_ends[k]] * 1000.0
                if k == g_idx:
                    ax1.axvspan(t_g_start, t_g_end, color='red', alpha=0.35, label=f'Selected Gap #{selected_gap}')
                else:
                    ax1.axvspan(t_g_start, t_g_end, color='gray', alpha=0.15)
            ax1.legend(loc='upper right')
        
        # Plot 2: Speed Trajectory & Duty Cycle Response
        ax2 = axs[1]
        ax2.plot(t_ms, rpm, label='Rotor Speed (RPM)', color='tab:green', linewidth=2)
        ax2.set_xlabel('Time (ms)')
        ax2.set_ylabel('Speed (RPM)', color='tab:green')
        ax2.set_title('Rotor Speed Response & Applied PWM Duty Cycle')
        ax2.grid(True, linestyle='--', alpha=0.6)
        
        ax2_duty = ax2.twinx()
        ax2_duty.plot(t_ms, duty, label='PWM Duty Cycle (%)', color='tab:red', linestyle='--', alpha=0.7)
        ax2_duty.set_ylabel('Duty Cycle (%)', color='tab:red')
        ax2_duty.set_ylim(-5, 105)
        
        if num_gaps > 0:
            t_g_start = res['t'][gap_starts[g_idx]] * 1000.0
            t_g_end = res['t'][gap_ends[g_idx]] * 1000.0
            ax2.axvspan(t_g_start, t_g_end, color='red', alpha=0.25, label=f'Readout Window #{selected_gap}')
        ax2.legend(loc='upper left')
        
        # Plot 3: Zoomed bEMF Gap & Edge Case Analysis
        ax3 = axs[2]
        if num_gaps > 0:
            g_s = gap_starts[g_idx]
            g_e = gap_ends[g_idx]
            t_start_ms = res['t'][g_s] * 1000.0
            t_end_ms = res['t'][g_e] * 1000.0
            
            # Analyse edge cases during selected gap
            g_current = res['i'][g_s:g_e+1]
            dt = res['t'][1] - res['t'][0] if len(res['t']) > 1 else 2e-6
            tail_duration_ms = np.sum(g_current > 0.0005) * dt * 1000.0
            peak_comm_spike = np.max(res['comm_spike'][g_s:g_e+1]) if 'comm_spike' in res else 0.0
            
            has_spike = peak_comm_spike > 0.05 * w_comm_spike_amp.value if w_comm_spike_amp.value > 0 else False
            has_tail_saturation = tail_duration_ms >= (w_bemf_window.value * 0.9)
            
            badges = []
            if has_spike:
                badges.append("<span style='color:red; font-weight:bold;'>⚡ COMMUTATION SPIKE OVERLAP</span>")
            if has_tail_saturation:
                badges.append("<span style='color:darkorange; font-weight:bold;'>⚠️ INDUCTIVE FLYBACK SATURATION</span>")
            if not badges:
                badges.append("<span style='color:green; font-weight:bold;'>✓ CLEAN READOUT WINDOW</span>")
                
            badge_str = " | ".join(badges)
            
            g_metric = res['gap_metrics'][g_idx] if ('gap_metrics' in res and g_idx < len(res['gap_metrics'])) else None
            measured_str = f" | Last 1/4 bEMF: {g_metric['measured_bemf']:.2f} V" if g_metric else ""
            pid_str = f" | PID Target: {w_target_bemf.value:.2f} V | Applied Duty: {g_metric['duty_applied']:.1f}%" if (g_metric and w_enable_pid.value) else ""
            
            gap_info_out.value = f"<b>Selected Gap #{selected_gap} of {num_gaps}:</b> t = {t_start_ms:.2f} ms – {t_end_ms:.2f} ms{measured_str}{pid_str} | Inductive Tail: {tail_duration_ms:.2f} ms | Peak Comm Spike: {peak_comm_spike:.3f} | {badge_str}"
            
            # Zoom mask (1ms padding before/after)
            zoom_mask = (res['t'] >= res['t'][g_s] - 0.001) & (res['t'] <= res['t'][g_e] + 0.002)
            
            t_zoom_ms = res['t'][zoom_mask] * 1000.0
            v_zoom = res['v_m'][zoom_mask]
            i_zoom = res['i'][zoom_mask]
            
            ax3.plot(t_zoom_ms, v_zoom, label='Terminal Voltage (V)', color='tab:purple', linewidth=2)
            ax3.plot(t_zoom_ms, i_zoom * 10, label='Current (x10 A)', color='tab:orange', linestyle='--')
            
            # Commutation spike indicator
            if 'comm_spike' in res and w_comm_spike_amp.value > 0:
                ax3_spike = ax3.twinx()
                spike_zoom = res['comm_spike'][zoom_mask]
                ax3_spike.plot(t_zoom_ms, spike_zoom, color='tab:red', linestyle=':', alpha=0.7, label='Comm Spike Amp')
                ax3_spike.set_ylabel('Comm Spike', color='tab:red')
                ax3_spike.tick_params(axis='y', labelcolor='tab:red')
            
            # Plot ADC sampled points & Highlight last 25% (Q4) readout window
            if len(res['adc_t']) > 0:
                adc_mask = (res['adc_t'] >= min(res['t'][zoom_mask])) & (res['adc_t'] <= max(res['t'][zoom_mask]))
                ax3.scatter(res['adc_t'][adc_mask] * 1000.0, res['adc_bemf'][adc_mask], color='red', zorder=5, label='ADC BEMF Sample (50kHz)')
                
            if g_metric is not None:
                q4_t_start_ms = g_metric['q4_start_t'] * 1000.0
                ax3.axvspan(q4_t_start_ms, t_end_ms, color='limegreen', alpha=0.2, label='Last 1/4 bEMF Window')
                if len(g_metric['q4_adc_t']) > 0:
                    ax3.scatter(g_metric['q4_adc_t'] * 1000.0, g_metric['q4_adc_bemf'], color='lime', edgecolors='black', zorder=6, label='Q4 Readout Samples')
                ax3.axhline(g_metric['measured_bemf'], color='green', linestyle='--', alpha=0.8, label=f"Measured bEMF ({g_metric['measured_bemf']:.2f}V)")
                
            if w_enable_pid.value:
                ax3.axhline(w_target_bemf.value, color='darkblue', linestyle=':', linewidth=2, label=f"PID Target bEMF ({w_target_bemf.value:.2f}V)")
                
            ax3.set_title(f'Zoomed Readout Gap #{selected_gap} (Window: {w_bemf_window.value:.1f} ms, Sampling Rate: {w_bemf_rate.value/1000:.0f} kHz)')
            ax3.set_xlabel('Time (ms)')
            ax3.set_ylabel('Voltage (V)')
            ax3.grid(True, linestyle='--', alpha=0.6)
            ax3.legend(loc='upper right')
        else:
            gap_info_out.value = "<b>Gap Status:</b> No bEMF readout gaps present in simulation duration."
            ax3.text(0.5, 0.5, 'No bEMF Gap Detected in Duration', ha='center', va='center')
            
        plt.tight_layout()
        plt.show()

def on_gap_change(change):
    run_and_plot_simulation()

w_gap_index.observe(on_gap_change, names='value')

btn_run = widgets.Button(description='Run Simulation', button_style='primary', icon='play')
btn_run.on_click(run_and_plot_simulation)

ui_box = widgets.VBox([
    preset_dropdown,
    description_out,
    widgets.HBox([
        widgets.VBox([widgets.HTML('<b>Operating Parameters</b>'), w_n_coils, w_v_in, w_pwm_freq, w_pwm_duty, w_pwm_duration, w_bemf_window, w_bemf_rate]),
        widgets.VBox([widgets.HTML('<b>Motor Model Parameters</b>'), w_R, w_L, w_Ke, w_J, w_ripple, w_comm_spike_amp, w_comm_spike_width, w_brush_noise]),
        widgets.VBox([widgets.HTML('<b>Closed-Loop PID Control (Last 1/4 bEMF)</b>'), w_enable_pid, w_target_bemf, w_kp, w_ki, w_kd])
    ]),
    widgets.HTML('<b>Readout Window & Edge Case Navigation</b>'),
    widgets.HBox([w_gap_index, btn_run]),
    gap_info_out,
    out
])

display(ui_box)
# Initial execution
run_and_plot_simulation()
